In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from deep_translator import GoogleTranslator

### Load the reports data

In [8]:
train = pd.read_csv("../data/train.csv")

print('Loaded the training set! Shape: ',train.shape)

Loaded the training set! Shape:  (4407, 14)


In [9]:
# Create a separate dataset to extract just the properly labeled studies
finding_cols = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
                 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
                 'Contusion', 'Fracture']

# gold is the 58 labeled studies
gold = train[train[finding_cols].notna().all(axis=1)]
print('The labeled part of the data has the shape: ', gold.shape)
gold.head(5)

The labeled part of the data has the shape:  (58, 14)


,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
29,1.2.826.0.1.3680043.8.498.10095687747295410396...,Antecedentes Clínicos:\nEsguince rodilla. [DAT...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
63,1.2.826.0.1.3680043.8.498.10170898615867673028...,The study reveals normal knee joint alignment...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
109,1.2.826.0.1.3680043.8.498.10306159113324811538...,Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
445,1.2.826.0.1.3680043.8.498.11287937729196958426...,"MRI of left Knee with \n-3-Plane Loc R'T, Sag ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
470,1.2.826.0.1.3680043.8.498.11382021393803389951...,"In the medial compartment, there is longitudi...",1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
!pip install deep-translator

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.0 MB/s eta 0:00:00

[notice] A new release of pip available: 22.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [10]:
# Translation function for reports

def translate_safe(text: str) -> str:
    if pd.isna(text) or text.strip() == '':
        return text
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except Exception:
        return None #flag failures to retry later

In [11]:
gold['Report_en'] = gold['Report'].apply(translate_safe)

/tmp/ipykernel_82550/1030815347.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gold['Report_en'] = gold['Report'].apply(translate_safe)


In [17]:
pd.set_option('display.max_colwidth', None)  # None = no limit

In [42]:
gold['Report_en'].head(10)

29                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [35]:
# This strips standard spaces, tabs, newlines, AND non-breaking spaces
gold['Report_en'] = gold['Report_en'].astype(str).str.strip(' \t\n\r\xa0')

/tmp/ipykernel_82550/4130679739.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gold['Report_en'] = gold['Report_en'].astype(str).str.strip(' \t\n\r\xa0')


In [41]:
# Look for '\xa0' or other hex codes in the output
print(gold['Report_en'][29])


Clinical History:
Knee sprain. [DATE].
Findings:
There are no significant bone marrow signal alterations.
Cruciate and collateral ligaments within normal limits.
Marginal amputation of the body of the lateral meniscus. Medial meniscus morphology and signal
preserved, without signs of breakage.
Cartilages of the femorotibial compartments without alterations.
Thin full-thickness focal chondral ulcer of the inferior aspect of the medial aspect of the trochlea
femoral with minimal secondary bone changes. Reparative chondral phenomena of the region
central of the femoral simplex. Patellar cartilage without alterations.
Mild joint effusion. There are no pathological popliteal cysts.
Increased signal of the distal insertion of the quadriceps tendon and the proximal insertion of the
patellar tendon, without signs of rupture.
There are no Hoffa fat signal alterations.
Printing:
Marginal amputation of the body of the lateral meniscus.
Grade 4 focal chondropathy of the inferior aspect of the medi

### Preprocessing the reports for tokenization / n-grams

The features have to separate *"medial meniscus tear"* from *"medial meniscus is
**not** torn"* — both reports talk about the same structure, but one is label 1
and the other label 0. Two consequences for this step:

1. **Negation is kept, not stopworded away.** A default `TfidfVectorizer` drops
   `no`, `not`, `without` as stopwords, which makes those two sentences almost
   identical. Here a negation cue scopes forward over its clause and the tokens
   it covers come out prefixed as `neg_`.
2. **Synonyms collapse to one token.** The reports are translations from several
   languages, so the same finding arrives as *tear / torn / rupture / breakage /
   disruption*. They map to a single `tear` token, and multi-word structures
   become one token (`medial_meniscus`) so a unigram already carries laterality.


In [ ]:
import re
import unicodedata

# --- 1. boilerplate that carries no finding -------------------------------
BOILERPLATE = [
    r"\[\s*(date|time|name|age|id|redacted)\s*\]",
    r"\b(exam|examination) (type|date and time|date)\s*:.*?(?=\n|$)",
    r"\btechnique\s*:.*?(?=\n\s*\n|$)",
    r"\bscan\s?protocol[^:]*:.*?(?=\n|$)",
    r"\bcomparison\s*:\s*no relevant prior.*?(?=\n|$)",
    r"\b\d+(\.\d+)?\s?(tesla|t)\b",
    r"\b(sag|ax|axial|cor|coronal|sagittal)\b[ /]*\b(t1|t2|pd|stir|fs|flair|gre)\b[\w /]*",
    r"\b\d-plane\s+loc[\w' ]*",
    r"\b(tr|te|ti|fov|nex)\s*[:=]?\s*\d+",
    r"\bsuggest correlation with clinical manifestation\b",
    r"\b(srs|img)\s*:\s*[\d\-,\s]+",
]

# Section headers come in two flavours. Administrative ones are noise and go
# away; anatomic ones name the subject of the sentence that follows them
# ("Medial meniscus: extensive tearing"), so only the colon is dropped - the
# words stay in the same clause to keep the n-gram intact.
ADMIN_HEADERS = (
    r"\b(clinical history|history|indication|findings|impression|printing|"
    r"conclusion|comparison|osseous structures|joint space|soft tissues|"
    r"extensor mechanism)\s*:"
)
ANATOMIC_HEADERS = (
    r"\b(medial|lateral|patellofemoral) (compartment|meniscus)( cartilage)?\s*:|"
    r"\b(cruciate|collateral) ligaments\s*:|"
    r"\b(anterior|posterior) cruciate ligament( \([apm]cl\))?\s*:"
)

# --- 2. domain vocabulary: many surface forms -> one token ----------------
# Order matters: longest / most specific first.
SYNONYMS = [
    # structures
    (r"\banterior cruciate ligaments?\b|\ba\.?c\.?l\.?\b", "acl"),
    (r"\bposterior cruciate ligaments?\b|\bp\.?c\.?l\.?\b", "pcl"),
    (r"\bmedial collateral ligaments?\b|\bm\.?c\.?l\.?\b|\btibial collateral ligaments?\b", "mcl"),
    (r"\b(lateral|fibular) collateral ligaments?\b|\bl\.?c\.?l\.?\b|\bf\.?c\.?l\.?\b", "lcl"),
    (r"\b(medial|internal) meniscus\b|\bmeniscus (medialis|internus)\b", "medial_meniscus"),
    (r"\b(lateral|external) meniscus\b|\bmeniscus (lateralis|externus)\b", "lateral_meniscus"),
    (r"\bmedial (femorotibial|femoro-tibial) (compartment|joint)?\b|\bmedial compartment\b", "medial_compartment"),
    (r"\blateral (femorotibial|femoro-tibial) (compartment|joint)?\b|\blateral compartment\b", "lateral_compartment"),
    (r"\bpatello-?femoral\b|\bretro-?patellar\b|\btrochlear?\b|\bpatellar facet\b|\bpatella\b", "patellofemoral"),
    # findings
    (r"\b(joint |articular |intra-?articular )?effusions?\b|\bfluid (accumulation|collection)\b|"
     r"\bfluid accumulated\b|\bjoint fluid\b|\bhydrarthrosis\b", "effusion"),
    (r"\bbakers? cysts?\b|\bpopliteal cysts?\b|\bcyst of baker\b", "bakers_cyst"),
    (r"\bsynovitis\b|\bsynovial (membrane )?(thickening|hypertrophy|proliferation)\b|"
     r"\bthicken\w* synovial tissue\b|\bsynovial reaction\b", "synovitis"),
    (r"\bbone (contusion|bruise|bruising)\b|\bcontusions?\b|\btrabecular (micro)?fracture\b", "contusion"),
    (r"\b(bone )?marrow (o?edema|signal alteration)\b|\bsubchondral (o?edema|marrow o?edema)\b|"
     r"\bbone o?edema\b", "marrow_edema"),
    (r"\b(avulsion |osteochondral |stress |acute |occult )?fractures?\b|\bfractured\b", "fracture"),
    (r"\bosteo-?arthriti[cs]\b|\bdegenerative (joint disease|arthropathy)\b|\bgonarthrosis\b|"
     r"\barthrosis\b", "osteoarthritis"),
    (r"\bosteophytes?\b|\bspurring\b|\bmarginal (osteophytes?|spurs?)\b", "osteophyte"),
    (r"\bchondromalacia\b|\bchondropathy\b|\bchondrosis\b|\bcartilage (loss|defect|thinning|wear)\b|"
     r"\bchondral (defect|ulcer|loss|lesion|fissur\w+)\b|\bcartilage (fissuring|heterogeneity)\b|"
     r"\bosteochondral defect\b", "chondral_loss"),
    (r"\bjoint space (narrowing|loss)\b|\bnarrowed joint space\b", "jsn"),
    (r"\btear(s|ing)?\b|\btorn\b|\bruptures?d?\b|\bbreakage\b|\bdisrupt\w*\b|\bdiscontinuit\w+\b|"
     r"\blacerations?\b|\bsplit\b", "tear"),
    (r"\bintact\b|\bpreserved\b|\bunremarkable\b|\bwithin normal limits\b|\bnormal\w*\b|"
     r"\bno abnormalit\w+\b", "normal"),
    (r"\bbursitis\b|\bdistended \w+ bursa\b", "bursitis"),
    (r"\btendinos\w+\b|\btendinopath\w+\b|\btendinitis\b", "tendinopathy"),
    (r"\bextrusion\b|\bextruded\b", "extrusion"),
    # severity / grade
    (r"\bhigh[- ]grade\b|\bgrade\s*(3|4|iii|iv)\b|\bfull[- ]thickness\b|\bcomplete\b|\bsevere\b|"
     r"\bextensive\b|\btotal\b", "sev_high"),
    (r"\blow[- ]grade\b|\bgrade\s*(1|2|i|ii)\b|\bpartial([- ]thickness)?\b|\bmild\b|\bminimal\b|"
     r"\bslight\b|\bsmall\b|\bmoderate\b|\bfocal\b", "sev_low"),
]

# --- 3. negation ----------------------------------------------------------
NEG_CUES = {
    "no", "not", "without", "absent", "absence", "denies", "negative",
    "free", "neither", "nor", "rules", "excluded", "unlikely", "resolved",
}
# scope ends here: a new clause starts
NEG_TERMINATORS = {
    "but", "however", "although", "though", "otherwise", "except", "while",
    "whereas", "there", "showing", "shows", "demonstrat",
}
NEG_WINDOW = 10

# --- 4. stopwords: generic English MINUS anything clinically load-bearing --
STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "in", "on", "at", "to", "for", "from",
    "by", "with", "is", "are", "was", "were", "be", "been", "being", "as", "it",
    "its", "this", "that", "these", "those", "there", "here", "his", "her",
    "their", "we", "he", "she", "they", "have", "has", "had", "do", "does",
    "did", "can", "could", "will", "would", "may", "might", "shall", "should",
    "seen", "noted", "note", "observed", "detected", "identified", "appears",
    "appear", "reveals", "reveal", "demonstrates", "demonstrate", "shows",
    "show", "study", "studies", "exam", "mri", "mr", "imaging", "image",
    "images", "series", "sequence", "sequences", "report", "patient", "knee",
    "right", "left", "both", "also", "well", "about", "approximately", "mm",
    "cm", "measuring", "measures", "size", "level", "region", "aspect", "area",
}

_MEAS = re.compile(r"\b\d+(\.\d+)?\s*[x×]\s*\d+(\.\d+)?\s*(mm|cm)?\b")
_NUM = re.compile(r"\b\d+(\.\d+)?\b")
_TOKEN = re.compile(r"[a-z_]+")


def preprocess_report(text, negate=True, keep_severity=True):
    """Normalise one radiology report for bag-of-words / n-gram modelling.

    Negation is preserved, not dropped: cues scope forward over the clause and
    the affected tokens come out as ``neg_<token>``, so "medial meniscus is not
    torn" and "medial meniscus tear" do not collapse to the same features.
    """
    if text is None or (isinstance(text, float) and text != text):
        return ""
    s = unicodedata.normalize("NFKC", str(text)).replace("\xa0", " ").lower()
    # translate_safe() returns None on failure; .astype(str) turns that into
    # the literal "none"/"nan", which would otherwise become a feature.
    if s.strip() in ("", "none", "nan"):
        return ""

    s = s.replace("\u2019", "'").replace("'", "")

    # anonymiser artefact: "0.9x0.4cm" came back as "intact9xintact4cm"
    s = re.sub(r"intact(\d)", r"0.\1", s)

    for pat in BOILERPLATE:
        s = re.sub(pat, " ", s, flags=re.S)
    s = re.sub(ADMIN_HEADERS, " . ", s)
    s = re.sub(ANATOMIC_HEADERS, lambda m: " " + m.group(0)[:-1] + " ", s)

    s = _MEAS.sub(" ", s)

    for pat, repl in SYNONYMS:
        s = re.sub(pat, f" {repl} ", s)

    s = _NUM.sub(" ", s)
    s = re.sub(r"[^a-z_\s\.;:\n]", " ", s)
    s = re.sub(r"[\.;:\n]+", " . ", s)
    s = re.sub(r"\s+", " ", s).strip()

    out, negating, since = [], False, 0
    for tok in s.split():
        if tok == ".":
            negating = False
            continue
        if tok in NEG_CUES:
            negating, since = True, 0
            out.append("neg_cue")
            continue
        if negating:
            since += 1
            if since > NEG_WINDOW or any(tok.startswith(t) for t in NEG_TERMINATORS):
                negating = False
        if tok in STOPWORDS:
            continue
        if not _TOKEN.fullmatch(tok):
            continue
        if not keep_severity and tok.startswith("sev_"):
            continue
        out.append(f"neg_{tok}" if negating else tok)
    return " ".join(out)

### Apply it to the labeled studies

In [ ]:
gold = gold.copy()          # gold was sliced out of train -> avoid SettingWithCopyWarning
gold['Report_clean'] = gold['Report_en'].map(preprocess_report)

print('empty after cleaning:', (gold['Report_clean'].str.len() == 0).sum(), 'of', len(gold))
print('mean tokens per report:', round(gold['Report_clean'].str.split().str.len().mean(), 1))

i = gold.index[0]
print('\n--- before ---\n', gold.loc[i, 'Report_en'][:400])
print('\n--- after ---\n', gold.loc[i, 'Report_clean'][:400])

### Vectorize

`token_pattern` has to be overridden: the default one splits on `_` and would
tear `medial_meniscus` and `neg_tear` back apart. Sublinear tf keeps a report
that repeats "effusion" five times from dominating one that says it once.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),          # "tear", "medial_meniscus tear", "neg_cue neg_tear"
    token_pattern=r"[a-z][a-z_]+",
    min_df=2,                    # a term seen in one report of 58 is noise
    sublinear_tf=True,
)
X = vectorizer.fit_transform(gold['Report_clean'])
y = gold[finding_cols].astype(int)

print('document-term matrix:', X.shape)
print('sample features:', list(vectorizer.get_feature_names_out()[::120])[:12])

### Sanity check — do the features agree with the labels?

Before training anything, check that the signal is actually there. This is a
deliberately dumb rule (is the finding's token present and un-negated, with a
supporting word nearby?), so whatever it scores is roughly a *floor* for a
trained model — if a label scores badly here, the vocabulary for it is missing.

In [ ]:
def _near(tokens, anchor, evidence, window=5):
    """Anchor token present with supporting evidence nearby, neither negated."""
    for i, tok in enumerate(tokens):
        if tok != anchor:
            continue
        lo, hi = max(0, i - window), min(len(tokens), i + window + 1)
        if any(t in evidence for t in tokens[lo:hi]):
            return 1
    return 0


RULES = {
    'ACL':              ('acl',              {'tear', 'sev_high', 'sprain', 'injury'}),
    'MCL':              ('mcl',              {'tear', 'sprain', 'injury', 'edema'}),
    'Medial Meniscus':  ('medial_meniscus',  {'tear', 'extrusion', 'degenerative'}),
    'Lateral Meniscus': ('lateral_meniscus', {'tear', 'extrusion', 'degenerative'}),
    'Medial OA':        ('medial_compartment',  {'chondral_loss', 'osteoarthritis', 'osteophyte', 'jsn'}),
    'Lateral OA':       ('lateral_compartment', {'chondral_loss', 'osteoarthritis', 'osteophyte', 'jsn'}),
    'PF OA':            ('patellofemoral',      {'chondral_loss', 'osteoarthritis', 'osteophyte', 'jsn'}),
    'Effusion':         ('effusion',    None),
    'Synovitis':        ('synovitis',   None),
    "Baker's":          ('bakers_cyst', None),
    'Contusion':        ('contusion',   None),
    'Fracture':         ('fracture',    None),
}

rows = []
for label, (anchor, evidence) in RULES.items():
    truth = gold[label].astype(int)
    pred = gold['Report_clean'].map(
        lambda c: _near(c.split(), anchor, evidence) if evidence else int(anchor in c.split())
    )
    rows.append({
        'label': label,
        'n_pos': int(truth.sum()),
        'agreement': (truth == pred).mean(),
        'missed (FN)': int(((truth == 1) & (pred == 0)).sum()),
        'spurious (FP)': int(((truth == 0) & (pred == 1)).sum()),
    })

check = pd.DataFrame(rows).sort_values('agreement')
print(check.to_string(index=False))
print('\nmacro agreement:', round(check['agreement'].mean(), 3))